# Merge LoRA adapter into Mistral base model and push to Hugging Face

This notebook shows the steps to: 
1. Install required libraries.
2. Load the base model and your LoRA adapter (from `mistral_adapter/checkpoint-*`).
3. Merge the LoRA weights into the base model and save the merged model.
4. Create a Hugging Face model repo and upload the merged model files.

Notes: You will need a Hugging Face token with write permission. Do NOT paste the token into public notebooks; set it as an environment variable or enter it when prompted. The merge step requires significant RAM/disk (the base Mistral model is large). Consider running on a machine with sufficient memory or use a cloud VM.

In [ ]:
# Install dependencies (run once). Uncomment to run here if needed.
# Note: this may take time and requires network access.
# !pip install -q transformers peft huggingface_hub accelerate safetensors

In [ ]:
import os
from getpass import getpass
from pathlib import Path

# Configure paths and model IDs
WORKDIR = Path('.')  # adjust if running from another folder
ADAPTER_DIR = WORKDIR / 'mistral_adapter' / 'checkpoint-256'  # update if different checkpoint
MERGED_DIR = WORKDIR / 'merged_mistral_lora'
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'
HF_REPO_ID = None  # set later or enter below (e.g. 'yourname/mistral-merged-lora')

print('Adapter dir:', ADAPTER_DIR)
print('Merged dir:', MERGED_DIR)

## Enter Hugging Face token and target repo
If you haven't created the Hugging Face repo yet, the script below can create it. Provide a repo name like `yourname/mistral-merged-lora`.

In [ ]:
# Get HF token securely
HF_TOKEN = os.getenv('HUGGINGFACE_TOKEN') or getpass('Hugging Face token (hf_...): ')
print('Token set? ', bool(HF_TOKEN))
# Optionally set repo id interactively
if HF_REPO_ID is None:
    HF_REPO_ID = input('Enter target repo (e.g. yourname/mistral-merged-lora): ').strip() or None
print('Target repo:', HF_REPO_ID)

## Merge LoRA into base model
This step loads the base model and applies the LoRA adapter from `ADAPTER_DIR`. It then merges the weights and saves the merged model to `MERGED_DIR`.
WARNING: This can require a lot of RAM and disk. If your machine doesn't have enough, run this on a cloud GPU instance.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MERGED_DIR.mkdir(parents=True, exist_ok=True)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

print('Loading base model (this may download several GB)...')
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

print('Loading LoRA adapter and wrapping via PeftModel...')
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR, device_map='auto')

print('Merging LoRA into base model (merge_and_unload)...')
try:
    peft_model = peft_model.merge_and_unload()  # returns a base model with weights merged (PEFT >=0.3 API may vary)
except Exception as e:
    # Some PEFT versions expose merge_and_unload as a method on PeftModel or require manual merging.
    print('merge_and_unload not available or failed:', e)
    # Try alternative: peft_model.base_model.save_pretrained after setting weights (riskier).

print('Saving merged model to', MERGED_DIR)
peft_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved.')

## Push merged model to Hugging Face Hub
This uploads the contents of `MERGED_DIR` to the repo `HF_REPO_ID`. If the repo doesn't exist, the script will create it.

In [ ]:
from huggingface_hub import HfApi, upload_folder, create_repo

api = HfApi()
if HF_REPO_ID is None:
    raise SystemExit('HF_REPO_ID is not set. Provide a repo id like yourname/mistral-merged-lora')

# Create repo if it doesn't exist (private by default)
try:
    api.create_repo(repo_id=HF_REPO_ID, private=True, token=HF_TOKEN)
    print('Created repo', HF_REPO_ID)
except Exception as e:
    print('Repo may already exist or creation failed:', e)

print('Uploading merged model folder to Hugging Face Hub (this may take several minutes)...')
upload_folder
(
    folder_path=str(MERGED_DIR),
    repo_id=HF_REPO_ID,
    path_in_repo='',
    token=HF_TOKEN,
)
print('Upload complete.')

## Next steps: deploy inference endpoint
- In the Hugging Face UI, go to your model repo → Deploy → Inference Endpoints → Create endpoint.
- Choose a GPU plan and point the endpoint at your newly uploaded merged model repo.
- Alternatively, create a custom container that loads the base model + adapter and expose a small POST API returning `image_prompt` and `caption`.

## Example request contract for your endpoint
Request (POST JSON): { "campaign_brief": "text here" }
Response (JSON): { "image_prompt": "...", "caption": "..." }

Once the endpoint is live, set `MISTRAL_CONTENT_MODE=endpoint` and `MISTRAL_ENDPOINT_URL` in your `.env`, restart the service, and test the chain.